# 00c · Quote repair check (no GPU)

Applies the quote repair to every failed decode saved by M0 and notebook 00b, and counts the
straight and curly double quotes in the five M1 credit agreements. Only straight quotes break
JSON when the model copies them unescaped.

**No GPU needed.** On Kaggle, set Accelerator to **None**. Internet must be **on**.
**Secrets:** `HF_TOKEN`. This notebook only reads: nothing is written to the Hub.

In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "m2-prompts"

In [ ]:
import os, subprocess, sys, site
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
with open(os.path.join(site.getsitepackages()[0], "credit_extractor.pth"), "w") as f:
    f.write(REPO + "\n")      # `import src` survives a kernel restart
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
# after a kernel restart, start again from this cell
from src.platform import pull, setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (M0 outputs are on the Hub)"

In [ ]:
import pandas as pd
from src import edgar, reproduce
cfg = reproduce.load_cfg()
plan = reproduce.prepare(env, cfg)      # pulls M0's records from the Hub
display(pd.DataFrame(reproduce.repair_check(plan)))

In [ ]:
m1 = edgar.load_cfg()["hub_prefix"]
pull(env.data_repo, m1, env.paths.work)
display(pd.DataFrame(edgar.quote_counts(env.paths.work / m1 / "text")))